# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


In [1]:
class matrix:
    def __init__(self, *args):
        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):
            n, m = args
            if n <= 0 or m <= 0:
                raise ValueError("Matrix dimensions must be positive integers.")
            self.n = n
            self.m = m
            self.data = [[0.0 for _ in range(m)] for _ in range(n)]
            
        elif len(args) == 1 and isinstance(args[0], list):
            data = args[0]
            if len(data) == 0 or not all(isinstance(row, list) for row in data):
                raise ValueError("Input must be a non-empty list of lists.")
            row_len = len(data[0])
            if row_len == 0 or not all(len(row) == row_len for row in data):
                raise ValueError("All rows must have the same non-zero number of columns.")
            self.n = len(data)
            self.m = row_len
            self.data = [[float(val) for val in row] for row in data]
        else:
            raise ValueError("Initialize with (n, m) or a list of lists.")

    def __repr__(self):
        return "\n".join([str(row) for row in self.data])

    def __getitem__(self, idx):
        if isinstance(idx, tuple):
            r, c = idx
            return self.data[r][c]
        return self.data[idx]

    def __setitem__(self, idx, value):
        if isinstance(idx, tuple):
            r, c = idx
            if isinstance(r, slice) and isinstance(c, slice):
                self.assign(value)
            else:
                self.data[r][c] = float(value)
        else:
            self.data[idx] = [float(x) for x in value]

    def assign(self, other):
        if isinstance(other, matrix):
            other_data = other.data
        elif isinstance(other, list):
            other_data = matrix(other).data  # validates shape
        else:
            raise TypeError("Can only assign from a matrix or a list of lists.")
        
        if len(other_data) != self.n or len(other_data[0]) != self.m:
            raise ValueError(f"Shape mismatch: cannot assign ({len(other_data)}, {len(other_data[0])}) to ({self.n}, {self.m}).")
        self.data = [row[:] for row in other_data]


M_zero = matrix(2, 3)
print("2x3 Zero Matrix:\n", M_zero)

M1 = matrix([[1, 2, 3], [4, 5, 6]])
print("\nM1[1][2] =", M1[1][2], "| M1[1, 2] =", M1[1, 2])

try:
    bad = matrix([[1, 2], [3]])
except ValueError as e:
    print("Caught expected invalid shape error:", e)

M2 = matrix(2, 3)
M2[:, :] = M1
print("\nM2 after assigning M1:\n", M2)

M2.assign([[7, 8, 9], [10, 11, 12]])
print("\nM2 after assigning list of lists:\n", M2)

try:
    M2.assign([[1, 2]])
except ValueError as e:
    print("Caught expected assignment error:", e)

2x3 Zero Matrix:
 [0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]

M1[1][2] = 6.0 | M1[1, 2] = 6.0
Caught expected invalid shape error: All rows must have the same non-zero number of columns.

M2 after assigning M1:
 [1.0, 2.0, 3.0]
[4.0, 5.0, 6.0]

M2 after assigning list of lists:
 [7.0, 8.0, 9.0]
[10.0, 11.0, 12.0]
Caught expected assignment error: Shape mismatch: cannot assign (1, 2) to (2, 3).


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

In [2]:
class matrix(matrix):
    def shape(self):
        return (self.n, self.m)

    def transpose(self):
        return matrix([[self.data[i][j] for i in range(self.n)] for j in range(self.m)])

    def row(self, i):
        return matrix([self.data[i][:]])

    def column(self, j):
        return matrix([[self.data[i][j]] for i in range(self.n)])

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        return matrix([[self.data[i][j] for j in range(n_0, n_1)] for i in range(m_0, m_1)])

    def __getitem__(self, idx):
        if isinstance(idx, tuple):
            r, c = idx
            if isinstance(r, slice) or isinstance(c, slice):
                rows = self.data[r] if isinstance(r, slice) else [self.data[r]]
                sub = [row[c] if isinstance(c, slice) else [row[c]] for row in rows]
                return matrix(sub)
            return self.data[r][c]
        elif isinstance(idx, slice):
            return matrix(self.data[idx])
        else:
            return self.data[idx]


M1 = matrix([[1, 2, 3], [4, 5, 6]])
print("Shape of M1:", M1.shape())
print("Transpose of M1:\n", M1.transpose())
print("Row 0 of M1:\n", M1.row(0))
print("Column 1 of M1:\n", M1.column(1))
print("to_list():", M1.to_list())
print("block(1, 3, 0, 2) [cols 1:3, rows 0:2]:\n", M1.block(1, 3, 0, 2))
print("Slicing M1[0:2, 1:3]:\n", M1[0:2, 1:3])

Shape of M1: (2, 3)
Transpose of M1:
 [1.0, 4.0]
[2.0, 5.0]
[3.0, 6.0]
Row 0 of M1:
 [1.0, 2.0, 3.0]
Column 1 of M1:
 [2.0]
[5.0]
to_list(): [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]
block(1, 3, 0, 2) [cols 1:3, rows 0:2]:
 [2.0, 3.0]
[5.0, 6.0]
Slicing M1[0:2, 1:3]:
 [2.0, 3.0]
[5.0, 6.0]


3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [3]:
def constant(n, m, c):
    return matrix([[float(c) for _ in range(m)] for _ in range(n)])

def zeros(n, m):
    return constant(n, m, 0.0)

def ones(n, m):
    return constant(n, m, 1.0)

def eye(n):
    return matrix([[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)])



print("constant(2, 2, 5):\n", constant(2, 2, 5))
print("\nzeros(2, 3):\n", zeros(2, 3))
print("\nones(2, 2):\n", ones(2, 2))
print("\neye(3):\n", eye(3))

constant(2, 2, 5):
 [5.0, 5.0]
[5.0, 5.0]

zeros(2, 3):
 [0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]

ones(2, 2):
 [1.0, 1.0]
[1.0, 1.0]

eye(3):
 [1.0, 0.0, 0.0]
[0.0, 1.0, 0.0]
[0.0, 0.0, 1.0]


4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

In [4]:
class matrix(matrix):
    def scalarmul(self, c):
        return matrix([[val * c for val in row] for row in self.data])

    def add(self, N):
        if self.shape() != N.shape():
            raise ValueError(f"Incompatible shapes for addition: {self.shape()} and {N.shape()}")
        return matrix([[self.data[i][j] + N.data[i][j] for j in range(self.m)] for i in range(self.n)])

    def sub(self, N):
        if self.shape() != N.shape():
            raise ValueError(f"Incompatible shapes for subtraction: {self.shape()} and {N.shape()}")
        return matrix([[self.data[i][j] - N.data[i][j] for j in range(self.m)] for i in range(self.n)])

    def mat_mult(self, N):
        if self.m != N.n:
            raise ValueError(f"Incompatible shapes for matrix multiplication: {self.shape()} and {N.shape()}")
        result = [[sum(self.data[i][k] * N.data[k][j] for k in range(self.m)) for j in range(N.m)] for i in range(self.n)]
        return matrix(result)

    def element_mult(self, N):
        if self.shape() != N.shape():
            raise ValueError(f"Incompatible shapes for element-wise multiplication: {self.shape()} and {N.shape()}")
        return matrix([[self.data[i][j] * N.data[i][j] for j in range(self.m)] for i in range(self.n)])

    def equals(self, N):
        if not isinstance(N, matrix) or self.shape() != N.shape():
            return False
        return all(self.data[i][j] == N.data[i][j] for i in range(self.n) for j in range(self.m))


X = matrix([[1, 2], [3, 4]])
Y = matrix([[5, 6], [7, 8]])

print("X.scalarmul(3):\n", X.scalarmul(3))
print("\nX.add(Y):\n", X.add(Y))
print("\nX.sub(Y):\n", X.sub(Y))
print("\nX.mat_mult(Y):\n", X.mat_mult(Y))
print("\nX.element_mult(Y):\n", X.element_mult(Y))
print("\nX.equals(X):", X.equals(X), "| X.equals(Y):", X.equals(Y))

try:
    X.add(matrix(3, 3))
except ValueError as e:
    print("Caught expected dimension error:", e)

X.scalarmul(3):
 [3.0, 6.0]
[9.0, 12.0]

X.add(Y):
 [6.0, 8.0]
[10.0, 12.0]

X.sub(Y):
 [-4.0, -4.0]
[-4.0, -4.0]

X.mat_mult(Y):
 [19.0, 22.0]
[43.0, 50.0]

X.element_mult(Y):
 [5.0, 12.0]
[21.0, 32.0]

X.equals(X): True | X.equals(Y): False
Caught expected dimension error: Incompatible shapes for addition: (2, 2) and (3, 3)


5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


In [5]:
class matrix(matrix):
    def __add__(self, other):
        return self.add(other)

    def __sub__(self, other):
        return self.sub(other)

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        elif isinstance(other, matrix):
            return self.mat_mult(other)
        raise TypeError("Unsupported operand type for *")

    def __rmul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        raise TypeError("Unsupported operand type for *")

    def __eq__(self, other):
        return self.equals(other)


M = matrix([[1, 2], [3, 4]])
N = matrix([[5, 6], [7, 8]])

print("2 * M:\n", 2 * M)
print("\nM * 2:\n", M * 2)
print("\nM + N:\n", M + N)
print("\nM - N:\n", M - N)
print("\nM * N:\n", M * N)
print("\nM == M:", M == M, "| M == N:", M == N)

2 * M:
 [2.0, 4.0]
[6.0, 8.0]

M * 2:
 [2.0, 4.0]
[6.0, 8.0]

M + N:
 [6.0, 8.0]
[10.0, 12.0]

M - N:
 [-4.0, -4.0]
[-4.0, -4.0]

M * N:
 [19.0, 22.0]
[43.0, 50.0]

M == M: True | M == N: False


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [6]:
A = matrix([[1, 2], [3, 4]])
B = matrix([[2, 0], [1, 2]])
C = matrix([[0, 1], [1, 0]])
I = eye(2)

print("(A * B) * C == A * (B * C):", ((A * B) * C) == (A * (B * C)))

print("A * (B + C) == A * B + A * C:", (A * (B + C)) == (A * B + A * C))

print("A * B != B * A:", (A * B) != (B * A))

print("A * I == A:", (A * I) == A)

(A * B) * C == A * (B * C): True
A * (B + C) == A * B + A * C: True
A * B != B * A: True
A * I == A: True
